# 教程 5：decode 时在线训练 —— 跨步与跨请求的训练目标

最后一站，我们做一件听起来有点激进的事：模型每吐出一个 token，就从自己的
生成过程里学一点。这不再是"先离线训练、再上线部署"的世界 —— 服务过程本身
就是训练过程。

不过真正的难点不在"训练"，而在"目标"。离线训练里，目标值和输入躺在同一个
batch 里，随手就能取；decode 途中的在线学习没这么奢侈 —— 你想学的目标，
很可能属于**别的时刻**。本篇里，我们的小头（一个 hidden→hidden 的映射 `w`）
要同时拟合三个目标：

1. **当前步**最后一层的 hidden —— 同层阶梯映射，最普通的一种；
2. **下一步**第 9 层的 hidden —— 这一步先写下自己的 h9，下一步拿到真实的
   h9 之后，回头校正自己上一步的预测；
3. **上一个 VIP 请求**（同一路由）结束时第 9 层的 hidden —— 目标干脆来自
   另一个请求。

前两个目标跨**步**，第三个跨**请求**。你大概会问：跨步的值从哪来？请求都
结束了，它的 hidden 怎么还在？答案只需要一对概念，你在教程 4 已经见过：

- **memory 字段**：`RequestState(..., memory={...})` 声明的普通字段 ——
  不参与优化、没有梯度与动量槽，但跟随训练一起发布；
- **捕获**：训练输入上的 `pllm.Capture("head", "prev_h9")` 表示：发布时把
  该输入的最后一行写进这个 memory 字段。

再加上一条熟悉的语义，一切就闭环了：memory 读到的是**输入版本** ——
第 N+1 步看到第 N 步发布的值。这就是"跨步"的全部含义；而 lineage 按 route
键跨请求存活，这就是"跨请求"的全部含义。没有新机制，只有旧积木的新组合。

## Workload：三个目标写在同一个 loss 里

先看 `learn` 的输入列表 —— 它就是三个目标的"采购清单"：

- `h9` 与 `h9_anchor` 是**同一个**第 9 层 hidden 读两次，各带一个 `Capture`：
  发布时分别落进 `prev_h9`（每步覆盖，留给下一步用）和 `anchor`（同样每步
  覆盖，请求结束时留在 memory 里的那个值就是终态）；
- `prev_h9` 与 `anchor` 再以 `Parameter` 来源各读一次 —— 读回的是 memory
  字段的**输入版本**，也就是上一步（或上一个请求）发布的值；
- 三个 loss 项一一对应三个目标：`loss_now` 拟合当前步最后一层，`loss_next`
  拿上一步的 h9 预测这一步的真值，`loss_vip` 向上一请求的终态对齐。

请注意：`learn` 里没有任何"目标"的概念。框架不知道、也不需要知道哪个张量
是标签 —— 目标只是从另一个时间点读进来的普通输入。返回值仍是老规矩：每个
状态一个标量，按状态求均值（state_mean）由 pllm 完成，函数只返回逐行 loss。

`predict` 在采样前运行，把 `anchor` 也加进引导（`beta=0.5`）：VIP 请求学到
的终态，直接参与后续请求的采样。训练影响生成，生成又产出新的训练材料 ——
循环就此闭合。

In [1]:
from typing import Annotated
import torch
import pllm
from vllm import SamplingParams
from pllm.config import VllmDType, ModelConfig


class DecodeTrainer(pllm.Workload):
    """decode 途中训练映射头 w，三类目标同时上线。"""

    head = pllm.RequestState(
        {"w": lambda m: (m.hidden_size, m.hidden_size)},
        dtype=torch.float32,
        optimizer=pllm.AdamW(lr=1e-3),
        memory={"prev_h9": lambda m: (m.hidden_size,),
                "anchor": lambda m: (m.hidden_size,)},
        keys=(pllm.state.RequestKeyField.ROUTE,))

    def ready(self, versions: Annotated[torch.Tensor, pllm.StateVersion("head")]) -> torch.Tensor:
        return versions > 0

    @pllm.on(pllm.Phase.BEFORE_SAMPLE, rows="ready")
    @pllm.guide_sampling(beta=0.5)
    def predict(self, h9: Annotated[torch.Tensor, pllm.Hidden(9)],
                w: Annotated[torch.Tensor, pllm.Parameter("head", "w")],
                anchor: Annotated[torch.Tensor, pllm.Parameter("head", "anchor")],
                versions: Annotated[torch.Tensor, pllm.StateVersion("head")]) -> torch.Tensor:
        return torch.bmm(h9.float().unsqueeze(1), w).squeeze(1) + 0.5 * anchor

    @pllm.on(pllm.Phase.AFTER_SAMPLE)
    @pllm.train(state="head", reduction="state_mean")
    def learn(self,
              h9: Annotated[torch.Tensor, pllm.Hidden(9), pllm.Capture("head", "prev_h9")],
              h9_anchor: Annotated[torch.Tensor, pllm.Hidden(9), pllm.Capture("head", "anchor")],
              h_last: Annotated[torch.Tensor, pllm.Hidden(-1)],
              w: Annotated[torch.Tensor, pllm.Parameter("head", "w")],
              prev_h9: Annotated[torch.Tensor, pllm.Parameter("head", "prev_h9")],
              anchor: Annotated[torch.Tensor, pllm.Parameter("head", "anchor")]) -> torch.Tensor:
        prediction = torch.bmm(h9.float().unsqueeze(1), w).squeeze(1)
        loss_now = (prediction - h_last.float()).square().mean(1)  # 目标 1：当前步最后一层
        loss_next = (torch.bmm(prev_h9.unsqueeze(1), w).squeeze(1)
                     - h9.float()).square().mean(1)                # 目标 2：上一步预测 → 这一步的真值
        loss_vip = (prediction - anchor).square().mean(1)          # 目标 3：向上一请求的终态对齐
        losses = loss_now + loss_next + loss_vip
        return losses  # 每行一个损失，pllm 按状态求均值

/workspace/pllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 三条通道，三次 generate 各看一条

说得好不如看得见。训练任务在 trace 选择器开启时会按步记录 `loss`，所以我们
设计三次生成，让每条通道各自露出马脚。先告诉你该期待什么：

1. **VIP**（`route="vip"`，全新 lineage）：首步的 `prev_h9` 和 `anchor`
   都是准入时零初始化的页，后续步开始读到自己上一步写下的捕获 —— 逐步
   loss 应当呈现一条明显变化的轨迹，而不是五个一样的数；
2. **续接**（同一个 `route="vip"`）：这条 lineage 还活着，首步就能读到
   VIP 留下的 memory 和训练过的 `w` —— 首步 loss 必须与一个全新 route
   不同，哪怕差别不大；
3. **两个全新 route**（`"fresh"` / `"fresh2"`）：同一 prompt、零状态，
   首步 loss 必须**完全相等**。这是零状态确定性检查，也是验证记忆通道
   最便宜的试金石。

下面的 `run` 用 trace 游标（`dropped + len(events)`）切出本次 generate 的
新鲜事件，只取 `loss` 张量 —— 教程 1 里见过的模式，原样复用。

In [2]:
from pllm.execution.trace import TraceSelection

llm = pllm.make_llm(workload=DecodeTrainer(), config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", VllmDType.BFLOAT16, 96, 4, 96, 0.2, seed=7))
plan = llm.attachment.plan
learn_index = next(i for i, task in enumerate(plan.semantic_graph.tasks)
                   if str(task.symbol) == "learn")
learn_id = next(task.task_id for task in plan.schedule_physical.tasks
                if task.task_id == learn_index)


def run(prompt, route):
    """一次 generate 的逐步 loss 与文本（trace 游标切片）。"""
    before = llm.trace_snapshot()
    cursor = before.dropped + len(before.events)
    llm.observe(TraceSelection(tasks=(learn_id,), tensors=True, max_tensor_bytes=32768))
    try:
        outputs = llm.generate([prompt], SamplingParams(max_tokens=6,
            temperature=0.0, ignore_eos=True), use_tqdm=False, routes=(route,))
        snapshot = llm.trace_snapshot()
        fresh = snapshot.events[max(0, cursor - snapshot.dropped):]
        losses = [(e.step_id, e.tensor().item()) for e in fresh if e.tensor_name == "loss"]
        return losses, outputs[0].outputs[0].text
    finally:
        llm.observe(TraceSelection())


vip_losses, vip_text = run("Important notice for alice@example.com: the VIP briefing", "vip")
print("VIP 逐步 loss:", [f"{v:.3f}" for _, v in vip_losses])
print("VIP 文本:", vip_text)

follow_losses, follow_text = run("Summarize the briefing", "vip")
fresh_losses, fresh_text = run("Summarize the briefing", "fresh")
fresh2_losses, _ = run("Summarize the briefing", "fresh2")
print("续接（同 route）首步 loss:", follow_losses[0][1], "文本:", follow_text)
print("新 route 首步 loss:", fresh_losses[0][1], "文本:", fresh_text)

# 跨步通道：VIP 首步的 prev_h9 是零，后续步读到自己上一步的捕获。
assert vip_losses[0][1] != vip_losses[-1][1]
# 跨请求通道：续接请求读到 VIP 的 memory 与训练累积，首步 loss 不同。
assert follow_losses[0][1] != fresh_losses[0][1]
# 零状态确定性：两个全新 route 的首步 loss 完全一致。
assert fresh_losses[0][1] == fresh2_losses[0][1]

d = llm.worker_state().diagnostics
print("losses:", d.loss_count, "updates:", d.update_count, "fallback:", d.fallback_count)
llm.close()

2026-09-10 18:43:28,500	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 09-10 18:43:29 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 96, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 96, 'max_num_seqs': 4, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 18:43:29 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 18:43:31 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 18:43:31 [model.py:1883] Using max model len 96


WARNING 09-10 18:43:31 [arg_utils.py:2620] This model does not officially support disabling chunked prefill. Disabling this manually may cause the engine to crash or produce incorrect outputs.


... (27 outputs left out)

INFO 09-10 18:43:42 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


WARNING 09-10 18:43:42 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _bmm_outer_product_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


VIP 逐步 loss: ['6.317', '5.714', '4.889', '5.649', '6.318']
VIP 文本:  is now available. The briefing


续接（同 route）首步 loss: 8.644431114196777 文本:  on the impact of climate change
新 route 首步 loss: 8.6561918258667 文本:  on the impact of climate change
losses: 20 updates: 20 fallback: 0


## 读懂输出

跑完对照三件事：

- **VIP 的逐步 loss 是一条轨迹**，首尾不相等 —— 跨步通道在工作。如果
  `prev_h9` 永远是零，`loss_next` 会把它钉死，逐步 loss 不会这样起伏；
- **续接与全新 route 的首步 loss 不同** —— 续接请求第一步就吃到了 VIP
  写下的 memory 与更新过的 `w`。差别可能很小，但一定存在；注意两者生成的
  文本完全一样，因为 `predict` 的引导差异还不足以翻转 argmax —— 状态差异
  先在 loss 上显形；
- **两个全新 route 的首步 loss 逐位相等** —— 同一 prompt、零初始状态，
  结果确定可复现。

最后的 diagnostics 给执行面盖章：`losses` 与 `updates` 计数一一对应（每步
一次 loss、一次更新），`fallback: 0` 表示没有任何操作逃出签名原语。

## 要点回顾

- 三类训练目标 = 三类输入来源：当前步 hidden、memory（上一步）、memory
  （上一请求）。框架里没有"目标"的概念 —— 目标只是从另一个时间点读进来的
  普通输入。
- `Capture` 写在训练输入上，发布时落进 memory；读取永远是输入版本 ——
  第 N+1 步看到第 N 步发布的值。
- 跨请求 = lineage 按 route 键跨请求存活，不引入任何请求侧的历史对象。
- 验证通道是否工作，最直接的办法是打开 trace 选择器看逐步 loss，再配合
  "全新 route 必须等于全新 route"的零状态确定性检查。

## 你已经走完的路

六篇教程，一篇一个核心动作：

- **教程 0**：`make_llm` 挂上你的第一个 workload，把数学挂进推理；
- **教程 1**：用 `@pllm.export` 与 trace 把 hidden 拿出来看，过一遍 SAE
  编码器；
- **教程 2**：写一个自己的训练头 —— loss 按状态返回，`@train` 授权更新，
  采样引导让训练影响生成；
- **教程 3**：给每个请求挂一副自己的 LoRA —— 请求级状态与 route 共享；
- **教程 4**：三种循环 —— replay 当步生效、Recurrence 跨步注入、会话循环
  跨请求传递；
- **教程 5**（本篇）：memory 与捕获把"训练目标"推广到任意时间点 —— 服务
  过程本身就是训练过程。

想继续深入：

- [训练与观测](../guides/training-and-observation.md) 按主题展开本篇用到的
  全部机制，`doc/guides/` 下还有状态、循环等专题；
- [创作契约](../reference/authoring-contract.md) 给出精确语义：update 掩码、
  detach 规则、行域 —— 写严肃 workload 前值得通读；
- [词汇表](../reference/glossary.md) 一个术语一个含义，读文档卡壳时回来
  查它。回 [README](../../README.md) 看全系列导航。